# Parse cached KO/EC TSVs → Parquet (issue #83)

Run **after** `scripts/download_to_cache.py` has populated `loaded_ko_ec/raw_cache/`.

Reads `loaded_ko_ec/manifest.csv` and the cached files on disk, parses
BLAST-style TSVs in **streaming batches**, and writes one Parquet per
`data_object_type`:

- `loaded_ko_ec/annotation_kegg_orthology.parquet`
- `loaded_ko_ec/annotation_enzyme_commission.parquet`

**Why streaming.** Each type holds ~1.4 B rows (~300 GB+ in pandas memory) —
the pod's cgroup memory cap is 24 GiB. So we use `pyarrow.parquet.ParquetWriter`
with a fixed schema, parse files in batches of ~500 MB raw TSV (~2 GB in
pandas), write each batch as a Parquet RowGroup, and drop. Peak memory stays
bounded regardless of total data volume.

**No network I/O.** Cache misses (files the downloader couldn't fetch) are
reported as errors at the end of each pass — no HTTP retries from the kernel.

When this notebook finishes, run `ingest_ko_ec_annotations.ipynb` to upload
to MinIO bronze + register Delta tables.


### Imports + config + logger

In [1]:
import io
import logging
import os
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Streaming batch target: parse this much raw TSV before flushing one
# Parquet RowGroup. Bigger batch = fewer row groups + slightly better
# compression; smaller batch = lower peak memory. 500 MB raw text expands
# to roughly 2 GB in pandas, comfortable inside the 24 GiB pod cap.
BATCH_TARGET_BYTES = int(os.environ.get("PARSE_BATCH_TARGET_BYTES", str(500 * 1024 * 1024)))

OUT_DIR = Path(os.environ.get("KO_EC_OUT_DIR", "loaded_ko_ec"))
if not OUT_DIR.exists():
    raise RuntimeError(f"OUT_DIR does not exist: {OUT_DIR.resolve()}")

CACHE_DIR = Path(os.environ.get("DOWNLOAD_CACHE_DIR", str(OUT_DIR / "raw_cache")))
if not CACHE_DIR.exists():
    raise RuntimeError(
        f"CACHE_DIR does not exist: {CACHE_DIR.resolve()}. "
        "Run scripts/download_to_cache.py first."
    )

LOG_DIR  = OUT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"parse_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("nmdc_lakehouse.ko_ec.parse")
logger.setLevel(logging.INFO)
for _h in list(logger.handlers):
    logger.removeHandler(_h)
_fh = logging.FileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(_sh)
logger.propagate = False

print(f"OUT_DIR:   {OUT_DIR.resolve()}")
print(f"CACHE_DIR: {CACHE_DIR.resolve()}")
print(f"LOG_FILE:  {LOG_FILE.resolve()}")


OUT_DIR:   /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec
CACHE_DIR: /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/raw_cache
LOG_FILE:  /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/logs/parse_20260429_185444.log


### Load the manifest

In [2]:
manifest_csv = OUT_DIR / "manifest.csv"
if not manifest_csv.exists():
    raise RuntimeError(
        f"manifest.csv not found at {manifest_csv.resolve()}. "
        "Run fetch_ko_ec_annotations.ipynb first."
    )
manifest = pd.read_csv(manifest_csv)
logger.info(f"manifest loaded: {len(manifest):,} rows from {manifest_csv}")
print(manifest.groupby("data_object_type").size().to_string())


manifest loaded: 9,696 rows from loaded_ko_ec/manifest.csv


data_object_type
Annotation Enzyme Commission    4848
Annotation KEGG Orthology       4848


### BLAST-style TSV parser (no header, 11 columns)

In [3]:
_PLACEHOLDER_PATTERNS = (
    "Nothing found in",
    "No KO Results for",
    "No EC Results for",
    "No KEGG Results for",
    "No Enzyme",
    "No annotations",
    "No Annotation",
)


def _is_placeholder(text: str) -> bool:
    stripped = text.strip()
    if not stripped:
        return True
    if "\n" in stripped:
        return False
    return any(stripped.startswith(p) for p in _PLACEHOLDER_PATTERNS)


# BLAST output schema, per issue #83 example:
#   gene_id  ncbi_taxid  KO:Kxxxxx  pct_identity  q_start  q_end  s_start  s_end  evalue  bitscore  aln_len
_BLAST_COLS = [
    "gene_id", "ncbi_taxid", "annotation_id",
    "pct_identity", "q_start", "q_end", "s_start", "s_end",
    "evalue", "bitscore", "aln_len",
]


def _parse_blast_tsv(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    df = pd.read_csv(
        io.StringIO(text),
        sep="\t",
        header=None,
        names=_BLAST_COLS,
        on_bad_lines="warn",
        engine="python",
    )
    if df.empty:
        return None
    return df


# Per-type parser hooks (identical implementation today, but separated so a
# future divergence in format is easy to slot in).
_PARSERS = {
    "Annotation KEGG Orthology":     _parse_blast_tsv,
    "Annotation Enzyme Commission":  _parse_blast_tsv,
}


### Cache reader + parse loop

In [4]:
def _cache_path_for(url: str) -> Path:
    return CACHE_DIR / urlparse(url).path.lstrip("/")


def _read_and_parse(row, parser):
    path = _cache_path_for(row.url)
    if not path.exists():
        return None, (row.url, "cache miss")
    try:
        text = path.read_text()
        df = parser(text)
        if df is None:
            return None, None  # placeholder / empty — not an error
        df["workflow_run_id"] = row.was_generated_by
        df["data_object_id"] = row.id
        return df, None
    except Exception as exc:
        return None, (row.url, str(exc))


# Fixed Arrow schema. Every batch is coerced to this before being written, so
# all RowGroups in the output Parquet share one schema.
ARROW_SCHEMA = pa.schema([
    ("gene_id",         pa.string()),
    ("ncbi_taxid",      pa.int64()),
    ("annotation_id",   pa.string()),
    ("pct_identity",    pa.float64()),
    ("q_start",         pa.int64()),
    ("q_end",           pa.int64()),
    ("s_start",         pa.int64()),
    ("s_end",           pa.int64()),
    ("evalue",          pa.float64()),
    ("bitscore",        pa.float64()),
    ("aln_len",         pa.int64()),
    ("workflow_run_id", pa.string()),
    ("data_object_id",  pa.string()),
])

_NUMERIC_INT_COLS   = ["ncbi_taxid", "q_start", "q_end", "s_start", "s_end", "aln_len"]
_NUMERIC_FLOAT_COLS = ["pct_identity", "evalue", "bitscore"]
_STRING_COLS        = ["gene_id", "annotation_id", "workflow_run_id", "data_object_id"]


def _coerce_for_arrow(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce columns to Arrow-compatible dtypes. Bad values become NA."""
    for col in _NUMERIC_INT_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in _NUMERIC_FLOAT_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in _STRING_COLS:
        # Use pandas StringDtype so NA in workflow_run_id round-trips cleanly.
        df[col] = df[col].astype("string")
    return df


def _flush(writer: pq.ParquetWriter, frames: list[pd.DataFrame]) -> int:
    """Concat, coerce, write one RowGroup. Returns rows written."""
    if not frames:
        return 0
    df = pd.concat(frames, ignore_index=True)
    df = _coerce_for_arrow(df)
    table = pa.Table.from_pandas(df, schema=ARROW_SCHEMA, preserve_index=False)
    writer.write_table(table)
    return len(df)


def stream_type(type_name: str, rows: pd.DataFrame, out_path: Path) -> dict:
    """Stream-parse all cached files for one data_object_type into one Parquet.

    Bounded memory: at most BATCH_TARGET_BYTES of raw TSV is held in pandas
    DataFrames at any time; each batch is concatenated, coerced to ARROW_SCHEMA,
    written as a Parquet RowGroup, and dropped.
    """
    parser = _PARSERS[type_name]
    stats = {"rows": 0, "files_with_data": 0, "placeholders": 0,
             "cache_misses": 0, "parse_errors": 0, "batches": 0}
    cache_misses_sample: list[str] = []
    parse_errors_sample: list[tuple[str, str]] = []

    total = len(rows)
    cache_hits = sum(1 for r in rows.itertuples() if _cache_path_for(r.url).exists())
    logger.info(f"  cache: {cache_hits:,}/{total:,} present; {total - cache_hits:,} missing")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        out_path.unlink()  # ParquetWriter cannot append to an existing file

    batch_frames: list[pd.DataFrame] = []
    batch_bytes = 0

    with pq.ParquetWriter(out_path, ARROW_SCHEMA, compression="snappy") as writer:
        for i, row in enumerate(rows.itertuples(), 1):
            df, err = _read_and_parse(row, parser)
            if df is not None:
                batch_frames.append(df)
                stats["files_with_data"] += 1
                cache_path = _cache_path_for(row.url)
                if cache_path.exists():
                    batch_bytes += cache_path.stat().st_size
            elif err is not None:
                if err[1] == "cache miss":
                    stats["cache_misses"] += 1
                    if len(cache_misses_sample) < 10:
                        cache_misses_sample.append(err[0])
                else:
                    stats["parse_errors"] += 1
                    if len(parse_errors_sample) < 10:
                        parse_errors_sample.append(err)
            else:
                stats["placeholders"] += 1

            if batch_bytes >= BATCH_TARGET_BYTES:
                n = _flush(writer, batch_frames)
                stats["rows"] += n
                stats["batches"] += 1
                logger.info(
                    f"    batch {stats['batches']}: {n:,} rows "
                    f"(file {i}/{total}, total rows so far {stats['rows']:,})"
                )
                batch_frames = []
                batch_bytes = 0

            if i % 500 == 0 or i == total:
                print(f"  {i}/{total}…", end="", flush=True)

        # final flush
        n = _flush(writer, batch_frames)
        if n:
            stats["rows"] += n
            stats["batches"] += 1
            logger.info(f"    batch {stats['batches']} (final): {n:,} rows")
    print()

    logger.info(
        f"  parsed: {stats['files_with_data']:,} files with data → "
        f"{stats['rows']:,} total rows in {stats['batches']:,} row groups; "
        f"{stats['placeholders']:,} placeholders; "
        f"{stats['cache_misses']:,} cache-misses; "
        f"{stats['parse_errors']:,} parse-errors"
    )
    if cache_misses_sample:
        for url in cache_misses_sample[:5]:
            logger.info(f"    cache-miss: {url}")
        if stats["cache_misses"] > 5:
            logger.info(f"    … and {stats['cache_misses'] - 5} more cache-misses")
    if parse_errors_sample:
        for url, msg in parse_errors_sample[:5]:
            logger.info(f"    parse-error: {url}: {msg}")
        if stats["parse_errors"] > 5:
            logger.info(f"    … and {stats['parse_errors'] - 5} more parse-errors")

    return stats


def _safe(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


### Run: one Parquet per `data_object_type`

In [ ]:
summary: dict[str, dict] = {}
t_run = time.monotonic()

for type_name, group in manifest.groupby("data_object_type"):
    out_path = OUT_DIR / f"{_safe(type_name)}.parquet"
    logger.info(f"\n{'─'*60}")
    logger.info(f"{type_name} ({len(group):,} files → {out_path.name})")

    stats = stream_type(type_name, group, out_path)
    if stats["rows"] == 0:
        logger.info("  no rows — removing empty Parquet")
        if out_path.exists():
            out_path.unlink()
        continue

    size_mb = out_path.stat().st_size / 1024**2
    logger.info(f"  wrote {out_path} ({size_mb:,.1f} MB)")
    summary[type_name] = stats

elapsed = time.monotonic() - t_run
logger.info(f"\n{'='*60}")
logger.info(f"Done in {elapsed/60:.1f} min")
for t, s in summary.items():
    logger.info(
        f"  {t}: {s['rows']:,} rows from {s['files_with_data']:,} files "
        f"({s['batches']:,} row groups)"
    )
logger.info(f"Full log: {LOG_FILE}")



────────────────────────────────────────────────────────────
Annotation Enzyme Commission (4,848 files → annotation_enzyme_commission.parquet)
  cache: 4,839/4,848 present; 9 missing
    batch 1: 5,405,818 rows (file 21/4848, total rows so far 5,405,818)
    batch 2: 5,429,838 rows (file 33/4848, total rows so far 10,835,656)
    batch 3: 6,380,997 rows (file 61/4848, total rows so far 17,216,653)
    batch 4: 6,365,129 rows (file 86/4848, total rows so far 23,581,782)
    batch 5: 6,487,936 rows (file 109/4848, total rows so far 30,069,718)
    batch 6: 7,028,305 rows (file 140/4848, total rows so far 37,098,023)
    batch 7: 5,805,378 rows (file 156/4848, total rows so far 42,903,401)
    batch 8: 5,892,817 rows (file 174/4848, total rows so far 48,796,218)
    batch 9: 5,440,878 rows (file 194/4848, total rows so far 54,237,096)
    batch 10: 6,265,561 rows (file 220/4848, total rows so far 60,502,657)
    batch 11: 6,020,869 rows (file 258/4848, total rows so far 66,523,526)
    b

  500/4848…

    batch 20: 5,527,261 rows (file 508/4848, total rows so far 120,259,386)
    batch 21: 5,377,832 rows (file 537/4848, total rows so far 125,637,218)
    batch 22: 6,342,356 rows (file 558/4848, total rows so far 131,979,574)
    batch 23: 5,335,353 rows (file 575/4848, total rows so far 137,314,927)
    batch 24: 5,647,061 rows (file 588/4848, total rows so far 142,961,988)
    batch 25: 5,500,764 rows (file 596/4848, total rows so far 148,462,752)
    batch 26: 5,341,055 rows (file 616/4848, total rows so far 153,803,807)
    batch 27: 6,539,602 rows (file 653/4848, total rows so far 160,343,409)
    batch 28: 5,691,308 rows (file 676/4848, total rows so far 166,034,717)
    batch 29: 5,694,313 rows (file 697/4848, total rows so far 171,729,030)
    batch 30: 5,616,307 rows (file 718/4848, total rows so far 177,345,337)
    batch 31: 5,507,020 rows (file 735/4848, total rows so far 182,852,357)
    batch 32: 6,766,240 rows (file 747/4848, total rows so far 189,618,597)
    batch 33

  1000/4848…

    batch 42: 5,779,063 rows (file 1002/4848, total rows so far 247,707,482)
    batch 43: 5,340,933 rows (file 1022/4848, total rows so far 253,048,415)
    batch 44: 5,682,332 rows (file 1033/4848, total rows so far 258,730,747)
    batch 45: 5,602,713 rows (file 1051/4848, total rows so far 264,333,460)
    batch 46: 5,381,242 rows (file 1083/4848, total rows so far 269,714,702)
    batch 47: 6,438,640 rows (file 1103/4848, total rows so far 276,153,342)
    batch 48: 5,812,435 rows (file 1137/4848, total rows so far 281,965,777)
    batch 49: 5,515,453 rows (file 1160/4848, total rows so far 287,481,230)
    batch 50: 5,479,607 rows (file 1186/4848, total rows so far 292,960,837)
    batch 51: 5,787,769 rows (file 1205/4848, total rows so far 298,748,606)
    batch 52: 6,035,157 rows (file 1230/4848, total rows so far 304,783,763)
    batch 53: 5,711,439 rows (file 1267/4848, total rows so far 310,495,202)
    batch 54: 6,080,581 rows (file 1291/4848, total rows so far 316,575,783)

...

  4848/4848…

    batch 207 (final): 6,118,164 rows
  parsed: 4,829 files with data → 1,231,453,377 total rows in 207 row groups; 10 placeholders; 9 cache-misses; 0 parse-errors
    cache-miss: https://data.microbiomedata.org/data/nmdc:omprc-11-x7sz0s85/nmdc:wfmgan-11-7ns8za11.1/nmdc_wfmgan-11-7ns8za11.1_ec.tsv
    cache-miss: https://data.microbiomedata.org/data/nmdc:omprc-11-g0vpnz53/nmdc:wfmgan-11-cfw6c937.1/nmdc_wfmgan-11-cfw6c937.1_ec.tsv
    cache-miss: https://data.microbiomedata.org/data/nmdc:omprc-11-sjb0gz53/nmdc:wfmgan-11-6smep943.1/nmdc_wfmgan-11-6smep943.1_ec.tsv
    cache-miss: https://data.microbiomedata.org/data/nmdc:omprc-11-5ghk8986/nmdc:wfmgan-11-64we1x93.1/nmdc_wfmgan-11-64we1x93.1_ec.tsv
    cache-miss: https://data.microbiomedata.org/data/nmdc:omprc-11-dkrhkz48/nmdc:wfmgan-11-cj1ts563.1/nmdc_wfmgan-11-cj1ts563.1_ec.tsv
    … and 4 more cache-misses
  wrote loaded_ko_ec/annotation_enzyme_commission.parquet (30,564.9 MB)

──────────────────────────────────────────────────────────

    batch 1: 7,618,463 rows (file 16/4848, total rows so far 7,618,463)
    batch 2: 6,599,062 rows (file 30/4848, total rows so far 14,217,525)
    batch 3: 6,975,842 rows (file 51/4848, total rows so far 21,193,367)
    batch 4: 5,473,202 rows (file 54/4848, total rows so far 26,666,569)
    batch 5: 7,432,646 rows (file 65/4848, total rows so far 34,099,215)
    batch 6: 7,546,696 rows (file 91/4848, total rows so far 41,645,911)
    batch 7: 5,850,908 rows (file 106/4848, total rows so far 47,496,819)
    batch 8: 6,558,732 rows (file 116/4848, total rows so far 54,055,551)
    batch 9: 5,644,442 rows (file 131/4848, total rows so far 59,699,993)
    batch 10: 6,610,082 rows (file 141/4848, total rows so far 66,310,075)
    batch 11: 5,702,806 rows (file 150/4848, total rows so far 72,012,881)
    batch 12: 7,067,584 rows (file 164/4848, total rows so far 79,080,465)
    batch 13: 6,457,775 rows (file 175/4848, total rows so far 85,538,240)
    batch 14: 5,530,986 rows (file 194/48

  500/4848…

    batch 33: 5,998,402 rows (file 519/4848, total rows so far 209,383,699)
    batch 34: 5,423,458 rows (file 538/4848, total rows so far 214,807,157)
    batch 35: 6,431,447 rows (file 544/4848, total rows so far 221,238,604)


### Spot-check

In [ ]:
for fname in ["annotation_kegg_orthology.parquet", "annotation_enzyme_commission.parquet"]:
    p = OUT_DIR / fname
    if not p.exists():
        print(f"-- {fname}: not written")
        continue
    df = pd.read_parquet(p)
    print(f"\n=== {fname}: {df.shape} ===")
    print(df.dtypes)
    print(df.head(3).to_string())
    print(f"distinct workflow runs: {df['workflow_run_id'].nunique():,}")
    print(f"distinct annotation_ids: {df['annotation_id'].nunique():,}")


## Next steps

Run **`ingest_ko_ec_annotations.ipynb`** to upload these Parquets to MinIO
bronze (`s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/results/`)
and register them as Delta tables under the `nmdc_results` namespace.
